# 04 - Analise estatistica e por regime

Carrega o CSV bruto gerado por `src/pipeline/run_all.py` e produz:
1. Tabela-resumo de metricas por modelo.
2. Diagrama de diferenca critica (Demsar 2006) via `autorank`.
3. Analise Bayesiana par a par com ROPE via `baycomp`.
4. Quebra dos resultados por regime (tamanho, numero de classes, proporcao categorica, missing).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd

from data.load_tabarena import summarize, RECOMMENDED_TASK_IDS
from src.reports.results_table import summary_by_model, pivot_for_stats
from src.pipeline.stats import demsar_analysis, bayesian_pairwise
from src.pipeline.regime import assign_regimes, aggregate_by_regime

In [2]:
raw = pd.read_csv('../results/raw.csv')
raw.head()

,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,359955,blood-transfusion-service-center,lightgbm,0.769006,0.791111,0.597537,0.466000,0.334508,0.014215,0.348724
1,359955,blood-transfusion-service-center,xgboost,0.727908,0.791111,0.632661,0.511380,1.467943,0.017070,1.485013
2,359955,blood-transfusion-service-center,catboost,0.704624,0.755556,0.557404,0.543412,2.244982,0.006570,2.251552
3,359955,blood-transfusion-service-center,group_model,0.758447,0.791111,0.558472,0.474704,0.406826,0.857361,1.264187
4,359968,churn,lightgbm,0.938793,0.955333,0.849317,0.151686,0.553601,0.016320,0.569921


In [3]:
summary_by_model(raw)

,model,auc_ovo_mean,auc_ovo_std,accuracy_mean,accuracy_std,g_mean_mean,g_mean_std,cross_entropy_mean,cross_entropy_std,total_time_s_mean,total_time_s_std
1,group_model,0.871638,0.096995,0.871907,0.088625,0.663090,0.290883,0.297446,0.170636,529.599407,981.826607
2,lightgbm,0.853298,0.103695,0.857472,0.094126,0.617019,0.290395,0.336361,0.171287,0.654392,0.355991
0,catboost,0.852080,0.100434,0.859491,0.090393,0.644086,0.250333,0.384254,0.207655,2.700736,1.105922
3,xgboost,0.849062,0.102937,0.858163,0.091837,0.620421,0.290265,0.422965,0.327900,3.113084,4.068336


In [5]:
pivot = pivot_for_stats(raw, metric='auc_ovo')
demsar = demsar_analysis(pivot, output_dir=Path('../results/figures'))
demsar['ranking']

             meanrank      mean       std ci_lower ci_upper effect_size  \
xgboost      3.233333  0.849062  0.102937      NaN      NaN         0.0   
catboost     2.900000  0.852080  0.100434      NaN      NaN   -0.029677   
lightgbm     2.733333  0.853298  0.103695      NaN      NaN   -0.041005   
group_model  1.133333  0.871638  0.096995      NaN      NaN   -0.225743   

              magnitude effect_size_above magnitude_above  
xgboost      negligible               0.0      negligible  
catboost     negligible         -0.029677      negligible  
lightgbm     negligible         -0.011937      negligible  
group_model       small         -0.182668      negligible  


group_model    1.133333
lightgbm       2.733333
catboost       2.900000
xgboost        3.233333
Name: mean_rank, dtype: float64

In [6]:
bayes = bayesian_pairwise(pivot, rope=0.01)
bayes

,model_a,model_b,p_a_worse,p_equivalent,p_a_better
0,catboost,group_model,0.00000,0.00340,0.99660
1,catboost,lightgbm,0.03166,0.92722,0.04112
2,catboost,xgboost,0.00610,0.99388,0.00002
3,group_model,lightgbm,0.93944,0.06056,0.00000
4,group_model,xgboost,0.99938,0.00062,0.00000
5,lightgbm,xgboost,0.10954,0.88460,0.00586


In [7]:
metadata = assign_regimes(summarize(RECOMMENDED_TASK_IDS))
for col in ['regime_size', 'regime_classes', 'regime_cat_share', 'regime_missing']:
    print(f'\n=== Agregado por {col} ===')
    print(aggregate_by_regime(raw, metadata, regime_col=col, metric_col='auc_ovo'))


=== Agregado por regime_size ===
   regime_size        model      mean       std  count
0        large     catboost  0.836429  0.089739      8
1        large  group_model  0.848803  0.086074      8
2        large     lightgbm  0.834139  0.092811      8
3        large      xgboost  0.833414  0.092481      8
4       medium     catboost  0.860152  0.102338     19
5       medium  group_model  0.881600  0.101421     19
6       medium     lightgbm  0.858923  0.110489     19
7       medium      xgboost  0.856098  0.106955     19
8        small     catboost  0.842687  0.148614      3
9        small  group_model  0.869441  0.121959      3
10       small     lightgbm  0.868766  0.117330      3
11       small      xgboost  0.846221  0.138895      3

=== Agregado por regime_classes ===
  regime_classes        model      mean       std  count
0         binary     catboost  0.838941  0.094191     24
1         binary  group_model  0.860989  0.087584     24
2         binary     lightgbm  0.842937  0.